# My Music Age — Stage 2: Data Pipeline

**Goal of this notebook:**
Read a Spotify listening-history file, clean it, and produce a well-structured
DataFrame that later stages (Music Age calculation, visualisation) can use.

**What it demonstrates (course concepts applied):**
- File handling (`open`, `json.load`) — *Module 1, L13*
- Pandas DataFrames, column operations, filtering — *Module 2, L20–22*
- String methods, regex for text cleaning — *Module 1, L4*
- `datetime` extraction for hour / day / month — *Module 2, L21*
- Groupby + aggregation for canonicalisation — *Module 2, L24*

This notebook depends on **`data_pipeline.py`** and **`generate_sample_data.py`**
sitting in the same folder.


## 1. Generate a sample Spotify export

Real Spotify data takes 1–30 days to arrive after you request it.
We generate a realistic fake file now so the entire pipeline can be
developed and tested. **Swapping in your real export later needs zero
code changes** — both files have the same schema.


In [1]:
# Run the generator module as a script.
# This creates StreamingHistory_sample.json in the current folder.
!python3 generate_sample_data.py



✓ Wrote 3,258 play records to StreamingHistory_sample.json
  Date range: 2025-04-23 03:13  →  2026-04-23 23:48
  Total listening: 9,913 minutes (165 hours)
  Unique artists: 30
  Unique tracks:  40


Peek at the first two records to confirm the format matches what Spotify gives us:


In [2]:
import json

with open("StreamingHistory_sample.json", encoding="utf-8") as f:
    sample = json.load(f)

print(f"Total records: {len(sample):,}")
print(f"First record:  {sample[0]}")
print(f"Second record: {sample[1]}")
print(f"Keys:          {list(sample[0].keys())}")


Total records: 3,258
First record:  {'endTime': '2025-04-23 03:13', 'artistName': 'Billie Eilish', 'trackName': 'What Was I Made For?', 'msPlayed': 163442}
Second record: {'endTime': '2025-04-23 05:42', 'artistName': 'Red Hot Chili Peppers', 'trackName': 'Under the Bridge', 'msPlayed': 282272}
Keys:          ['endTime', 'artistName', 'trackName', 'msPlayed']


## 2. Load the file into a DataFrame

The `load_listening_history()` function accepts **both** Spotify export
schemas (basic and extended) and our sample file. It returns a DataFrame
with four standardised columns: `end_time`, `artist`, `track`, `ms_played`.


In [3]:
from data_pipeline import load_listening_history

raw = load_listening_history("StreamingHistory_sample.json")
print(f"Loaded {len(raw):,} records")
raw.head()


Loaded 3,258 records


,end_time,artist,track,ms_played
0,2025-04-23 03:13:00,Billie Eilish,What Was I Made For?,163442
1,2025-04-23 05:42:00,Red Hot Chili Peppers,Under the Bridge,282272
2,2025-04-23 12:57:00,Radiohead,Creep,174640
3,2025-04-23 19:26:00,Arctic Monkeys,Do I Wanna Know?,265168
4,2025-04-23 20:46:00,The Weeknd,Save Your Tears,94907


In [4]:
# Quick sanity check — what are the column types?
raw.dtypes


end_time     datetime64[us]
artist                  str
track                   str
ms_played             int64
dtype: object

In [5]:
# And the date range?
print(f"From: {raw['end_time'].min()}")
print(f"To:   {raw['end_time'].max()}")


From: 2025-04-23 03:13:00
To:   2026-04-23 23:48:00


## 3. Clean the raw data

The `clean_listening_history()` function performs five operations:

1. **Drop missing rows** — any record without timestamp, artist, track or duration
2. **Normalise strings** — strip whitespace, lowercase the matching keys
3. **Drop skip-throughs** — plays under 30 seconds (default) are usually accidental
4. **Drop exact duplicates** — Spotify sometimes includes them across re-exports
5. **Canonicalise display names** — so "taylor swift" and "Taylor Swift" merge to the common form
6. **Extract derived time columns** — `hour`, `day_of_week`, `month`, `year`, `minutes_played`

It also adds two *key* columns (`artist_key`, `track_key`) that we'll use
to merge with the Kaggle catalogue in Stage 3.


In [6]:
from data_pipeline import clean_listening_history

clean = clean_listening_history(raw)
print(f"\nFinal shape: {clean.shape}")
clean.head()


  cleaned: 3,258 → 3,013 rows (dropped 245, 7.5%)

Final shape: (3013, 11)


,end_time,artist,track,ms_played,artist_key,track_key,hour,day_of_week,month,year,minutes_played
0,2025-04-23 03:13:00,Billie Eilish,What Was I Made For?,163442,billie eilish,what was i made for?,3,Wednesday,4,2025,2.724033
1,2025-04-23 05:42:00,Red Hot Chili Peppers,Under the Bridge,282272,red hot chili peppers,under the bridge,5,Wednesday,4,2025,4.704533
2,2025-04-23 12:57:00,Radiohead,Creep,174640,radiohead,creep,12,Wednesday,4,2025,2.910667
3,2025-04-23 19:26:00,Arctic Monkeys,Do I Wanna Know?,265168,arctic monkeys,do i wanna know?,19,Wednesday,4,2025,4.419467
4,2025-04-23 20:46:00,The Weeknd,Save Your Tears,94907,the weeknd,save your tears,20,Wednesday,4,2025,1.581783


In [7]:
# All the columns we now have, with their purposes:
for col, dtype in zip(clean.columns, clean.dtypes):
    print(f"  {col:15s}  {str(dtype)}")


  end_time         datetime64[us]
  artist           str
  track            str
  ms_played        int64
  artist_key       str
  track_key        str
  hour             int32
  day_of_week      str
  month            int32
  year             int32
  minutes_played   float64


## 4. Summarise the listening history

The `summarise_listening()` helper returns a dict of headline stats. These
same numbers will feed the "hero banner" of the final report.


In [8]:
from data_pipeline import summarise_listening

summary = summarise_listening(clean)
for k, v in summary.items():
    print(f"  {k:15s}  {v}")


  plays            3013
  unique_artists   30
  unique_tracks    40
  total_minutes    9850.7
  total_hours      164.2
  date_range       (Timestamp('2025-04-23 03:13:00'), Timestamp('2026-04-23 23:48:00'))
  top_artist       Arctic Monkeys
  top_track        Save Your Tears


## 5. Quick exploratory checks

Let's verify the data "feels right" before moving to Stage 3. These
are not part of the final report — just confidence checks.


### 5.1  Top 10 artists by play count


In [9]:
top_artists = (
    clean.groupby("artist")
         .size()
         .sort_values(ascending=False)
         .head(10)
)
top_artists


artist
Arctic Monkeys           527
The Weeknd               512
Taylor Swift             474
Billie Eilish            272
Queen                    207
Linkin Park              190
Coldplay                 147
Michael Jackson          147
Red Hot Chili Peppers     65
Radiohead                 61
dtype: int64

### 5.2  Top 10 tracks by total listening time


In [10]:
top_tracks = (
    clean.groupby(["artist", "track"])["minutes_played"]
         .sum()
         .sort_values(ascending=False)
         .head(10)
         .round(1)
)
top_tracks


artist           track            
The Weeknd       Save Your Tears      1118.2
Arctic Monkeys   Do I Wanna Know?     1063.8
Taylor Swift     Anti-Hero             815.2
                 Cruel Summer          770.9
Billie Eilish    lovely                717.6
Arctic Monkeys   R U Mine?             634.3
The Weeknd       Blinding Lights       560.2
Queen            Don't Stop Me Now     353.7
Linkin Park      In The End            335.5
Michael Jackson  Thriller              329.9
Name: minutes_played, dtype: float64

### 5.3  Listening by hour of day

A realistic listening pattern should peak in the evenings. If our
sample data is well-simulated, we should see that shape here.


In [11]:
hourly = (
    clean.groupby("hour")["minutes_played"]
         .sum()
         .round(1)
)
print(hourly)


hour
0       94.5
1      131.6
2      118.7
3      127.2
4      159.2
5      139.7
6      274.0
7      456.8
8      503.2
9      394.6
10     218.6
11     339.4
12     406.6
13     347.4
14     253.4
15     248.6
16     379.3
17     497.8
18     773.4
19     980.8
20    1035.7
21     927.8
22     688.8
23     353.6
Name: minutes_played, dtype: float64


### 5.4  Listening by day of week


In [12]:
# Order the days properly (default alphabetical order is wrong)
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday",
             "Friday", "Saturday", "Sunday"]

daily = (
    clean.groupby("day_of_week")["minutes_played"]
         .sum()
         .reindex(day_order)
         .round(1)
)
print(daily)


day_of_week
Monday       1151.7
Tuesday      1212.6
Wednesday    1273.3
Thursday     1251.7
Friday       1166.3
Saturday     1949.0
Sunday       1846.0
Name: minutes_played, dtype: float64


## 6. Next step — Stage 3

The `clean` DataFrame is now ready to be joined with a Kaggle Spotify
catalogue (1.2M+ tracks with release years) so we can calculate **Music Age**.

That's Stage 3. Nothing in Stage 3 needs to know how Stage 2 produced its
DataFrame — the only contract between the two is the column set:
`end_time, artist, track, ms_played, artist_key, track_key, hour, day_of_week, month, year, minutes_played`.
